In [0]:
from pyspark.sql import functions as F

catalogo = "medalhao"
gold_db_name = "gold"
silver_db_name = "silver"

#criar os df a partir das tabelas do silver
df_itens = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos")
df_traducao = spark.table(f"{catalogo}.silver.dim_categoria_produtos_traducao").select(F.col("nome_produto_pt").alias("categoria_produto"),
                                                                           F.col("nome_produto_en").alias("categoria_produto_en")) 
df_pedido_total = spark.table(f"{catalogo}.silver.fat_pedido_total").select("id_pedido", "data_pedido", "valor_total_pago_brl", "valor_total_pago_usd") #mantem apenas essas tabelas
df_cotacao = spark.table(f"{catalogo}.silver.dim_cotacao_dolar")

df_itens_enriquecido = (df_itens
    .join(df_produtos, on="id_produto", how="left")
    .join(df_traducao, on="categoria_produto", how="left")
    .join(df_pedido_total.select("id_pedido", "data_pedido"), on="id_pedido", how="left") 
    .join(df_cotacao, F.col("data_pedido") == F.col("data"), how="left")) #enriquecendo os itens com os dados de produtos, traducao e pedido total

display(df_itens_enriquecido.count())

df_itens_enriquecido = (df_itens_enriquecido.withColumn("ano_venda", F.year(F.col("data_pedido")))
                                            .withColumn("mes_venda", F.month(F.col("data_pedido")))) #extraindo mes e ano do pedido

#calcular métricas
fat_vendas_comercial = (df_itens_enriquecido.groupBy("ano_venda", "mes_venda", "categoria_produto") #agrupa todos os dados por mes, ano e categoria
                        .agg(F.countDistinct("id_pedido").alias("total_pedidos"), #evita duplicar os pedidos
                            F.count("id_item").alias("qtd_itens_vendidos"), #conta todas as linhas e mostra o volume real vendido
                            F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"), #soma o preco em BRL de todos os itens e arredonda para 2 casas decimais
                            F.round(F.sum(F.col("preco_BRL") / F.col("cotacao_compra_usd")), 2).alias("receita_total_usd"),  #divide o preco em BRl pela cotacao para ter a receita em USD
                            F.round(F.sum("preco_BRL") / F.countDistinct("id_pedido"), 2).alias("ticket_medio_brl")).orderBy("ano_venda", "mes_venda", "categoria_produto")) #divide a receita pela qtd de pedidos para ter o ticket medio em BRL e ordena por ano, mes e cateforia

(fat_vendas_comercial.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.gold.fat_vendas_comercial"))

fat_vendas_comercial.display()

In [0]:
df_mais_vendidos = (df_itens_enriquecido.groupBy("nome_produto", "categoria_produto")
                    .agg(F.count("id_item").alias("quantidade_vendida"))
                    .orderBy(F.col("quantidade_vendida").desc()).limit(5)) #agrupa por produto e categoria e ordena por quantidade vendida (5 mais vendidos)

df_mais_vendidos.display()

df_menos_vendidos = (df_itens_enriquecido.groupBy("nome_produto", "categoria_produto")
                    .agg(F.count("id_item").alias("quantidade_vendida"))
                    .orderBy(F.col("quantidade_vendida").asc()).limit(5)) #agrupa por produto e categoria e ordena por quantidade vendida (5 menos vendidos)

df_menos_vendidos.display()



In [0]:
df_avaliacoes = spark.table(f"{catalogo}.silver.fat_avaliacoes_pedidos")
df_vendedores = spark.table(f"{catalogo}.silver.dim_vendedores").select("id_vendedor", "cidade_vendedor", "estado_vendedor", "nome_vendedor")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos").select("id_produto", "categoria_produto", "nome_produto")

#enquicer os dados, dando um leftjoin nas tres tabelas para agrupar depois de acordo com a regra de negocio
df_avaliacoes_enriquecidas = (df_avaliacoes.join(df_itens.select("id_pedido", "id_produto", "id_vendedor"), on="id_pedido", how="left")
                                            .join(df_produtos, on="id_produto", how="left")
                                            .join(df_vendedores, on="id_vendedor", how="left")
)

#agrupando por categoria, id_vendedor e estado do vendedor e calculando as metricas
fat_avaliacoes_clientes = (df_avaliacoes_enriquecidas.groupBy("categoria_produto", "id_vendedor", "estado_vendedor","nome_vendedor","nome_produto", "nota_avaliacao", "id_avaliacao")
                              .agg(F.count("id_avaliacao").alias("total_avaliacoes"), #conta o total de avaliacoes (quantidade de linhas de id_avaliacao)
                                   F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"), #avg calcula a media aritmetrica da tabela com 2 casas decimais
                                   F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"), #condicao para contar as avaliacoes positivas, tranformas todas as linhas em 1 se a nota for >= a 4 e trasnforma as linhas em 0 caso contrário
                                   F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas"),) #condicao para contar as avaliacoes negativas, tranformas todas as linhas em 1 se a nota for <= a 2 e trasnforma as linhas em 0 caso contrário
                                  .withColumn("percentual_satisfacao", F.when(F.col("total_avaliacoes") > 0, F.round(F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes") * 100, 2)).otherwise(0)))
                                  
(fat_avaliacoes_clientes.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.gold.fat_avaliacoes_clientes"))

fat_avaliacoes_clientes.display()


In [0]:
df_melhor_produto =(fat_avaliacoes_clientes
.groupBy("nome_produto","categoria_produto")
.agg(
        F.round(F.avg("nota_avaliacao"),2).alias("nota_media"), #f.round arredonda o valor da nota_media para duas casas decimais
        F.count("id_avaliacao").alias("total_avaliacoes") #f.count conta o total de avaliacoes
)
.orderBy(F.col("nota_media").desc(), F.col("total_avaliacoes").desc())
.limit(1)
) #agrupando o melhor produto pelo nome e categoria e ordenando por nota_media e total_avaliacoes, ordenando a coluna nota_media em ordem decrescente e total_avaliacoes em ordem decrescente

print("Produto melhor avaliado")
df_melhor_produto.display()

df_pior_produto =(fat_avaliacoes_clientes
.groupBy("nome_produto","categoria_produto")
.agg(
        F.round(F.avg("nota_avaliacao"),2).alias("nota_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
)
.orderBy(F.col("nota_media").asc(), F.col("total_avaliacoes").desc())
.limit(1)
)#agrupando o pior produto pelo nome e categoria e ordenando por nota_media e total_avaliacoes, ordenando a coluna nota_media em ordem crescente e total_avaliacoes em ordem decrescente

print("Produto pior avaliado")	
df_pior_produto.display()

df_melhor_vendedor =(fat_avaliacoes_clientes
.groupBy("nome_vendedor","estado_vendedor")
.agg(
        F.round(F.avg("nota_avaliacao"),2).alias("nota_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
)
.orderBy(F.col("nota_media").desc(), F.col("total_avaliacoes").desc())
.limit(1)
) #agrupando o melhor vendedor pelo nome e estado e ordenando por nota_media e total_avaliacoes, ordenando a coluna nota_media em ordem decrescente e total_avaliacoes em ordem decrescente

print("Vendedor melhor avaliado")
df_melhor_vendedor.display()

df_pior_vendedor =(fat_avaliacoes_clientes
.groupBy("nome_vendedor","estado_vendedor")
.agg(
        F.round(F.avg("nota_avaliacao"),2).alias("nota_media"), 
        F.count("id_avaliacao").alias("total_avaliacoes")
)
.orderBy(F.col("nota_media").asc(), F.col("total_avaliacoes").desc())
.limit(1)
)#agrupando o pior vendedor pelo nome e estado e ordenando por nota_media e total_avaliacoes, ordenando a coluna nota_media em ordem crescente e total_avaliacoes em ordem decrescente

print("Vendedor pior avaliado")
df_pior_vendedor.display()